# 05 — Create a Bedrock Managed Knowledge Base with Google Drive Connector

This notebook creates a BMKB using the **Google Drive** connector to index documents stored in Google Drive.

**What "Bedrock Managed" means:** Bedrock handles the vector store for you. The Google Drive connector syncs files from your Drive folders and indexes their content — no S3 upload needed.

### What this notebook does

1. Configures your environment (region, Google Drive settings, Secrets Manager)
2. Creates an AWS Secrets Manager secret with your Google OAuth2 credentials
3. Creates the BMKB with IAM roles and policies (auto-managed by utility)
4. Creates a Google Drive data source with inclusion/exclusion filters
5. Runs ingestion (syncs documents from Google Drive)
6. Queries the KB using Retrieve and AgenticRetrieveStream
7. Cleans up all resources

### Prerequisites

- AWS credentials with Bedrock, IAM, and Secrets Manager permissions
- Enable model access for your embedding and generation models in the [Amazon Bedrock console](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access-modify.html)
- **Google Cloud OAuth2 credentials** — `clientId`, `clientSecret`, and `refreshToken`
- Google Drive API enabled in your Google Cloud project

### Architecture

```
Google Drive ──► BMKB Connector ──► BMKB (Bedrock-managed vector store) ──► Retrieve / AgenticRetrieveStream
     │                  │                    │
     ├── My Drive       ├── Secrets Manager  ├── IAM Role (auto-created)
     ├── Shared Drives  ├── OAuth2 creds     ├── Embedding: Managed default (no extra cost)
     └── Folders         └── Filters          └── Parsing: Smart Parsing
```

### Google Drive Authentication

| Component | Description |
|-----------|-------------|
| OAuth2 Client | Google Cloud OAuth2 client ID and secret |
| Refresh Token | Long-lived token to access Drive on behalf of a user |
| Secrets Manager | Stores the OAuth2 credentials securely in AWS |

### Reference

- [Amazon Bedrock Knowledge Bases — Google Drive connector](https://docs.aws.amazon.com/bedrock/latest/userguide/googledrive.html)

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

Set your region, KB name, Google Drive settings, and model preferences.

In [ ]:
import boto3
import json
import sys
import time
import logging
import pprint

try:
    from dotenv import load_dotenv; load_dotenv('../../.env')
except ImportError:
    pass

sys.path.insert(0, "../..")

# Clients
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# Generate unique suffix
suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration (update these) ─────────────────────────────────────
knowledge_base_name = f'bmkb-googledrive-{suffix}'
knowledge_base_description = 'BMKB workshop - Google Drive data source'

# ┌─────────────────────────────────────────────────────────────────────┐
# │  GOOGLE DRIVE CREDENTIALS — UPDATE THE 3 VALUES BELOW              │
# │                                                                     │
# │  Replace the placeholder text between the quotes with your actual   │
# │  Google OAuth2 credentials. Keep the quotes around the values.      │
# │                                                                     │
# │  Example:                                                           │
# │    google_client_id = '123456789.apps.googleusercontent.com'        │
# │    google_client_secret = 'GOCSPX-AbCdEfGhIjKlMnOpQrStUv'         │
# │    google_refresh_token = '1//0abc123...'                           │
# └─────────────────────────────────────────────────────────────────────┘

google_client_id     = 'YOUR_CLIENT_ID'       # ← paste your Client ID here
google_client_secret = 'YOUR_CLIENT_SECRET'   # ← paste your Client Secret here
google_refresh_token = 'YOUR_REFRESH_TOKEN'   # ← paste your Refresh Token here

# Secret name in AWS Secrets Manager (will be created if it doesn't exist)
secret_name = f'bmkb-googledrive-secret-{suffix}'

# ── Models ─────────────────────────────────────────────────────────────
# Embedding model — use None for managed default (no extra cost)
embedding_model = None  # Managed default
# embedding_model = 'amazon.titan-embed-text-v2:0'  # Custom — additional cost
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:      {region}')
print(f'Account:     {account_id}')
print(f'KB Name:     {knowledge_base_name}')
print(f'Secret:      {secret_name}')
print(f'Embedding:   {embedding_model or "Managed default (no extra cost)"}')
print(f'Generation:  {generation_model_arn}')

## Step 2 — Create the Secrets Manager Secret

The Google Drive connector requires an AWS Secrets Manager secret containing OAuth2 credentials. This secret will be referenced by the connector when syncing documents.

**Secret format expected by Bedrock:**
```json
{
  "clientId": "your-oauth2-client-id",
  "clientSecret": "your-oauth2-client-secret",
  "refreshToken": "your-refresh-token"
}
```

In [ ]:
secretsmanager_client = boto3.client('secretsmanager', region_name=region)

google_credentials = {
    'clientId': google_client_id,
    'clientSecret': google_client_secret,
    'refreshToken': google_refresh_token,
}

# Validate — all fields must be set
placeholder_values = {'YOUR_CLIENT_ID', 'YOUR_CLIENT_SECRET', 'YOUR_REFRESH_TOKEN'}
missing = [k for k, v in google_credentials.items() if v in placeholder_values or not v]
if missing:
    raise ValueError(f'Update the following in Step 1 before running: {missing}')

# Create (or update) the secret in Secrets Manager
try:
    response = secretsmanager_client.create_secret(
        Name=secret_name,
        Description='Google Drive OAuth2 credentials for BMKB connector',
        SecretString=json.dumps(google_credentials),
    )
    secret_arn = response['ARN']
    print(f'Created secret: {secret_arn}')
except secretsmanager_client.exceptions.ResourceExistsException:
    # Update existing secret
    secretsmanager_client.put_secret_value(
        SecretId=secret_name,
        SecretString=json.dumps(google_credentials),
    )
    response = secretsmanager_client.describe_secret(SecretId=secret_name)
    secret_arn = response['ARN']
    print(f'Updated existing secret: {secret_arn}')

## Step 3 — Create the Bedrock Managed Knowledge Base with Google Drive

The unified `ManagedKnowledgeBase` utility handles everything:
- Creates IAM execution role with scoped policies (model + CloudWatch + Secrets Manager)
- Creates a `MANAGED` type KB
- Creates a Google Drive data source with your configuration
- Waits for all resources to become active

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    data_sources=[{
        'type': 'GOOGLEDRIVE',
        'secret_arn': secret_arn,
        # What to crawl (all enabled by default):
        # 'data_entity_config': {
        #     'type': 'DRIVE',
        #     'crawlMyDrive': True,
        #     'crawlSharedWithMe': True,
        #     'crawlSharedDrives': True,
        # },
        # Optional: restrict to specific folders
        # 'inclusion_filters': ['folder_id_1', 'folder_id_2'],
        # 'exclusion_filters': ['folder_id_to_skip'],
    }],
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest (Sync Google Drive documents)

Start an ingestion job. The connector will sync accessible documents from the Google Drive folders shared with the service account and index their content.

In [ ]:
job = kb.start_ingestion_job()

## Step 5 — Query the Knowledge Base

### 5a. Retrieve API — Raw Chunks

In [ ]:
# Update query to match your Google Drive documents
response = kb.retrieve('What are the key topics covered in the documents?', num_results=5)

print('=== Retrieve API ===')
for i, res in enumerate(response.get('retrievalResults', []), 1):
    score = res['score']
    text = res['content']['text'][:120]
    print(f'{i}. score={score:.4f} | {text}...')
print(f'\nTotal: {len(response.get("retrievalResults", []))} chunks')

### 5b. AgenticRetrieveStream (with generation)

In [ ]:
result = kb.agentic_retrieve_stream(
    query='Summarize the main findings from the documents.',
    model_arn=generation_model_arn,
    generate_response=True,
)

print('=== AgenticRetrieveStream (with generation) ===')
print(f"Answer:\n{result['generated_response']['answer']}")
print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")
for i, citation in enumerate(result['generated_response'].get('citations', [])[:3], 1):
    refs = citation.get('references', [])
    start = citation.get('startIndex', 0)
    end = citation.get('endIndex', 0)
    print(f'  Citation {i} [chars {start}-{end}]: {len(refs)} reference(s)')
    for ref in refs[:2]:
        idx = ref.get('resultIndex', -1)
        if idx >= 0 and idx < len(result['results']):
            print(f'    - result[{idx}]: {result["results"][idx]["content"]["text"][:80]}...')

### 5c. AgenticRetrieveStream — Query Decomposition

In [ ]:
result = kb.agentic_retrieve_stream(
    query='What are the different topics discussed and how do they relate to each other?',
    model_arn=generation_model_arn,
    max_results=10,
    max_iterations=3,
)

print('=== AgenticRetrieveStream API ===')
print(f'Trace events: {len(result["traces"])}')
for t in result['traces']:
    attrs = t.get('attributes', {})
    print(f'  [{attrs.get("step", "?")}] {attrs.get("status", "")}')

print(f'\nFinal: {len(result["results"])} deduplicated chunks')
for i, r in enumerate(result['results'][:5], 1):
    text = r['content']['text'][:120]
    print(f'  {i}. {text}...')
# Show generated response if available
if result.get('generated_response'):
    print(f"\nGenerated Answer:\n{result['generated_response']['answer']}")
    print(f"Citations: {len(result['generated_response'].get('citations', []))}")

### 5d. Try your own queries

In [ ]:
# Update query to match your Google Drive documents
QUERY = 'What is the most important recommendation in the documents?'

result = kb.agentic_retrieve_stream(
    query=QUERY,
    model_arn=generation_model_arn,
    generate_response=True,
)
print(result['generated_response']['answer'])

## Step 6 — Cleanup

Delete all resources. The utility handles correct ordering:
data sources → KB → IAM roles/policies.

You also need to delete the Secrets Manager secret separately.

> Only run this when you're done experimenting.

In [ ]:
# Uncomment to delete everything
print('===============================Deleting Knowledge Base and associated resources==============================')
# kb.delete_kb(delete_iam=True)

# Delete the Secrets Manager secret
# secretsmanager_client.delete_secret(
#     SecretId=secret_name,
#     ForceDeleteWithoutRecovery=True,
# )
# print(f'Deleted secret: {secret_name}')

In [ ]:
# Alternative: delete by KB ID only
# from utils.managed_knowledge_base import ManagedKnowledgeBase
# ManagedKnowledgeBase.delete_kb_by_id('YOUR_KB_ID', region_name='us-west-2')

## Summary

| What | Details |
|---|---|
| KB Type | `MANAGED` — Bedrock handles the vector store |
| Data Source | Google Drive via `MANAGED_KNOWLEDGE_BASE_CONNECTOR` |
| Authentication | Google OAuth2 credentials (clientId, clientSecret, refreshToken) stored in Secrets Manager |
| Parsing | Smart Parsing (default, managed by Bedrock) |
| Embedding | Managed default (no extra cost) or custom |
| IAM | Auto-created scoped role + policies (model, CloudWatch, Secrets Manager) |
| Retrieval | `Retrieve` (raw chunks), `AgenticRetrieveStream` (agentic retrieval with optional generation) |

### Google Drive Setup Checklist

| Step | Where | Description |
|---|---|---|
| Enable Drive API | Google Cloud Console | APIs & Services → Library → Google Drive API |
| Create OAuth2 Client | Google Cloud Console | APIs & Services → Credentials → OAuth 2.0 Client IDs |
| Generate Refresh Token | OAuth2 Playground or your app | Authorize with `drive.readonly` scope |
| Store credentials | AWS Secrets Manager | Created by this notebook (Step 2) |